# 이미지 설명 요청하기

## 인터넷에 있는 이미지 사진 설명 요청하기
- 원하는 이미지 링크 사용
- 단, API 호출을 통해 서버에 접근 가능한 이미지 링크를 사용하기

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPEN_API_KEY")  # 환경 변수에서 API 키를 가져오기

client = OpenAI(api_key=api_key)  # 오픈AI 클라이언트의 인스턴스 생성

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "이 이미지에 대해 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://recipe1.ezmember.co.kr/cache/recipe/2016/06/05/1ba7e34cf0daf694f09a3a9539ebdb161.jpg",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response


ChatCompletion(id='chatcmpl-CQOyCJRvkhaWfHrI5WzVsOcqvK8OJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='이 이미지는 오므라이스입니다. 계란으로 감싼 볶음밥 위에 케첩을 뿌려 장식이 되어 있습니다. 접시에는 브로콜리와 케첩, 과일 조각도 함께 담겨 있습니다. 볶음밥 속에는 잘게 썬 햄과 채소가 들어 있는 것으로 보입니다.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))], created=1760408548, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_cbf1785567', usage=CompletionUsage(completion_tokens=83, prompt_tokens=779, total_tokens=862, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

## 내가 가진 이미지로 설명 요청하기

### base64 인코딩
- 바이너리 데이터 HTTP Request 전송을 위해 인코딩 진행

In [3]:
import base64

# Function to encode the image
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
image_path = "./data/mart.jpg"

# 이미지를 base64로 인코딩
base64_image = encode_image(image_path)

print(base64_image[0:100])

/9j/4AAUSkZJRgABAQEBLAEsAABBTVBG/+EJxEV4aWYAAE1NACoAAAAIAA0BDwACAAAABgAAAKoBEAACAAAACgAAALABEgADAAAA


### GPT API 호출 및 결과 출력

In [4]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "이 이미지에 대해 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response.choices[0].message.content

'이 이미지는 가게의 선반을 보여주고 있습니다. 선반에는 다양한 인스턴트 라면과 컵라면이 진열되어 있습니다. 왼쪽에는 봉지 라면이 쌓여있고, 오른쪽 선반에는 컵라면들이 여러 개 놓여 있습니다. 아래쪽에는 면 같은 식재료가 포장되어 진열되어 있습니다. 전체적으로 인스턴트 식품 코너로 보입니다.'

### 여러 이미지 비교 분석 요청

In [5]:
red_dumpling_base64 = encode_image("data/red_dumpling.jpeg")
dumpling_soup_base64 = encode_image("data/dumpling_soup.jpeg")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "두 음식의 차이점을 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{red_dumpling_base64}",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{dumpling_soup_base64}",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response.choices[0].message.content

'두 음식 모두 만두 요리로 보이나, 조리 방식과 소스의 차이가 있습니다.\n\n첫 번째 음식:\n- 소스가 붉은색인 것으로 보아 매운 소스를 사용한 것 같습니다.\n- 만두는 소스에 담겨 있거나 버무려져 있어 풍미가 강할 것으로 예상됩니다.\n\n두 번째 음식:\n- 맑은 국물에 담겨 있으며, 국물 요리에 가까운 형태입니다.\n- 만두 외에 계란 지단 등이 추가되어 있어 부드럽고 담백한 맛을 낼 것으로 보입니다.\n\n두 음식의 주된 차이는 소스와 국물이 다르다는 점입니다. 하나는 매운 소스를, 다른 하나는 맑은 국물을 사용하여 맛과 식감이 크게 다를 수 있습니다.'

## GPT 비전의 한계 알아보기

In [6]:
oecd_rnd_2021_base64 = encode_image("./data/oecd_rnd_2021_large.png")
oecd_rnd_2022_base64 = encode_image("./data/oecd_rnd_2022_large.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "첫번째는 2021년 데이터이고, 두번째는 2022년 데이터입니다. 이 데이터에 대해 설명해주세요. 어떤 변화가 있었나요? 한국 중심으로 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{oecd_rnd_2021_base64}",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{oecd_rnd_2022_base64}",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response.choices[0].message.content


'두 데이터는 2021년과 2022년의 연구개발비와 GDP 대비 연구개발비 비중을 보여주고 있습니다. 한국을 중심으로 살펴보면 다음과 같은 변화가 있습니다:\n\n### 연구개발비\n\n- **2021년:** 89,282백만 US달러\n- **2022년:** 91,013백만 US달러\n\n한국의 연구개발비가 2021년에서 2022년으로 증가했습니다.\n\n### GDP 대비 연구개발비 비중\n\n- **2021년:** 4.93%\n- **2022년:** 4.91%\n\nGDP 대비 연구개발비 비중은 약간 감소했습니다.\n\n### 종합\n\n한국의 연구개발비 자체는 증가했으나, GDP 대비 비중이 미세하게 감소하였습니다. 이는 GDP성장이 연구개발비 증가속도보다 빨랐음을 시사할 수 있습니다. \n\n한국의 연구개발 투자에 대한 지속적인 관심과 확대 노력이 확인되는 한편, GDP대비 비중에서의 감소는 경제 전반의 다른 부문 성장의 영향일 수 있습니다.'

In [7]:
oecd_rnd_2021_base64 = encode_image("./data/oecd_rnd_2021_medium.png")
oecd_rnd_2022_base64 = encode_image("./data/oecd_rnd_2022.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "첫번째는 2021년 데이터이고, 두번째는 2022년 데이터입니다. 이 데이터에 대해 설명해주세요. 어떤 변화가 있었나요? 한국 중심으로 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{oecd_rnd_2021_base64}",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{oecd_rnd_2022_base64}",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response.choices[0].message.content

'두 데이터는 2021년과 2022년 각국의 연구 개발비(R&D)와 GDP 대비 연구개발비 비중을 보여줍니다. 한국을 중심으로 살펴보면 다음과 같은 변화가 있습니다.\n\n### 1. 연구개발비 변화\n- **2021년**: 한국의 연구개발비는 약 121,739백만 달러입니다.\n- **2022년**: 한국의 연구개발비는 약 133,867백만 달러로 증가했습니다.\n\n### 2. GDP 대비 연구개발비 비중 변화\n- **2021년**: 한국의 GDP 대비 연구개발비 비중은 4.93%입니다.\n- **2022년**: 한국의 GDP 대비 연구개발비 비중은 5.21%로 증가했습니다.\n\n### 요약\n- 한국은 2021년에서 2022년 사이에 연구개발비와 GDP 대비 비중 모두 증가했습니다. 이는 한국이 R&D에 대한 투자를 지속적으로 확대하고 있음을 보여줍니다.\n- 이러한 성장은 OECD 국가들 중에서도 높은 수준이며, 한국의 과학 기술에 대한 집중적인 투자를 나타냅니다.\n\n이러한 변화를 통해 한국이 과학기술 선진국으로서의 위치를 유지하고자 하는 노력을 확인할 수 있습니다.'